In [58]:
import pandas as pd

df_train = pd.read_csv("train.csv")
df_test = pd.read_csv("test.csv")

df_train.head()

,Patient_ID,Age,Gender,Symptoms,Symptom_Count,Disease
0,P0786,22,Male,"fever, cough, sneezing, runny nose",4,Common Cold
1,P0097,86,Male,"increased thirst, fatigue",2,Diabetes
2,P0257,13,Other,"headache, sensitivity to light, nausea",3,Migraine
3,P0763,38,Other,"fever, fatigue",2,Pneumonia
4,P0895,25,Other,"blurred vision, increased thirst",2,Diabetes


In [59]:
#preprocessing

genders = pd.get_dummies(df_train.Gender, dtype=int) #one hot encoding gender
df_train = df_train.join(genders).drop(columns=['Gender'])

genders = pd.get_dummies(df_test.Gender, dtype=int)
df_test = df_test.join(genders).drop(columns=['Gender'])

from sklearn.preprocessing import LabelEncoder #encoding disease
enc = LabelEncoder()
df_train.Disease = enc.fit_transform(df_train.Disease)

from sklearn.preprocessing import MultiLabelBinarizer #encoding list of categorical features brrrrrrrrrrrr

def make_list(text):
    return text.split(", ")

df_train.Symptoms = df_train.Symptoms.apply(make_list)
df_test.Symptoms = df_test.Symptoms.apply(make_list)

mlb = MultiLabelBinarizer()
symptoms_train = pd.DataFrame(mlb.fit_transform(df_train.Symptoms), columns=mlb.classes_)
symptoms_test = pd.DataFrame(mlb.transform(df_test.Symptoms), columns=mlb.classes_)

df_train = df_train.join(symptoms_train).drop(columns=['Symptoms'])
df_test = df_test.join(symptoms_test).drop(columns=['Symptoms'])

df_train.head()

,Patient_ID,Age,Symptom_Count,Disease,Female,Male,Other,blurred vision,chest pain,chest tightness,...,muscle pain,nausea,rash,runny nose,sensitivity to light,sensitivity to sound,shortness of breath,sneezing,sore throat,wheezing
0,P0786,22,4,3,0,1,0,0,0,0,...,0,0,0,1,0,0,0,1,0,0
1,P0097,86,2,4,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,P0257,13,3,8,0,0,1,0,0,0,...,0,1,0,0,1,0,0,0,0,0
3,P0763,38,2,9,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,P0895,25,2,4,0,0,1,1,0,0,...,0,0,0,0,0,0,0,0,0,0


In [60]:
df_train.columns

Index(['Patient_ID', 'Age', 'Symptom_Count', 'Disease', 'Female', 'Male',
       'Other', 'blurred vision', 'chest pain', 'chest tightness', 'chills',
       'cough', 'dizziness', 'fatigue', 'fever', 'frequent urination',
       'headache', 'increased thirst', 'itchy eyes', 'loss of taste',
       'muscle pain', 'nausea', 'rash', 'runny nose', 'sensitivity to light',
       'sensitivity to sound', 'shortness of breath', 'sneezing',
       'sore throat', 'wheezing'],
      dtype='object')

In [61]:
features = ['Age', 'Symptom_Count', 'Female', 'Male', 'Other', 'blurred vision', 'chest pain', 'chest tightness', 'chills',
       'cough', 'dizziness', 'fatigue', 'fever', 'frequent urination', 'headache', 'increased thirst', 'itchy eyes', 'loss of taste',
       'muscle pain', 'nausea', 'rash', 'runny nose', 'sensitivity to light', 'sensitivity to sound', 'shortness of breath', 'sneezing', 'sore throat', 'wheezing']
target = 'Disease'

X = df_train[features]
y = df_train[target]

X_final = df_test[features]

X.head()

,Age,Symptom_Count,Female,Male,Other,blurred vision,chest pain,chest tightness,chills,cough,...,muscle pain,nausea,rash,runny nose,sensitivity to light,sensitivity to sound,shortness of breath,sneezing,sore throat,wheezing
0,22,4,0,1,0,0,0,0,0,1,...,0,0,0,1,0,0,0,1,0,0
1,86,2,0,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,13,3,0,0,1,0,0,0,0,0,...,0,1,0,0,1,0,0,0,0,0
3,38,2,0,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,25,2,0,0,1,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [62]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

model = LogisticRegression(max_iter=10000)

cross_val_score(model, X, y, cv=5, scoring='accuracy')

array([0.9125 , 0.95625, 0.9    , 0.90625, 0.9    ])

In [63]:
model.fit(X, y)
y_preds = model.predict(X_final)
y_preds[:5]

array([5, 7, 9, 8, 4])

In [64]:
y_preds = enc.inverse_transform(y_preds)
y_preds[:5]

array(['Heart Disease', 'Influenza', 'Pneumonia', 'Migraine', 'Diabetes'],
      dtype=object)

In [65]:
submission = pd.DataFrame({
    "Patient_ID": df_test.Patient_ID,
    "Disease": y_preds
})

submission

,Patient_ID,Disease
0,T0001,Heart Disease
1,T0002,Influenza
2,T0003,Pneumonia
3,T0004,Migraine
4,T0005,Diabetes
...,...,...
195,T0196,Asthma
196,T0197,COVID-19
197,T0198,Hypertension
198,T0199,Hypertension


In [68]:
submission.to_csv("submission.csv", index=False)